# 📊 The Receiver Operating Characteristic (ROC) Curve

Welcome to the hands-on explanation notebook for the **ROC Curve**! In this notebook, we will:
1. Define the mathematical coordinates of the ROC Curve (True Positive Rate vs. False Positive Rate).
2. Implement the ROC calculation from scratch using NumPy and verify it against `scikit-learn`.
3. Simulate predictions for a **Good Model** (high class separation) vs. a **Poor Model** (overlapping classes).
4. Plot the ROC Curves for both models to visualize how separation affects the bowing towards the top-left corner.
5. Contrast ROC Curves with Precision-Recall (PR) Curves and explain why PR curves are preferred in object detection systems like YOLO.

Let's start by importing the necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Set seed for reproducibility
np.random.seed(42)

## 1. Data Generation

We simulate two classifiers:
1.  **Good Classifier:** Predicts high scores for Class 1 (mean 0.8) and low scores for Class 0 (mean 0.2).
2.  **Poor Classifier:** Predicts highly overlapping scores (mean 0.55 for Class 1, mean 0.45 for Class 0).

In [ ]:
n_samples = 150
y_true = np.concatenate([np.ones(75), np.zeros(75)]).astype(int)

# Good Classifier Scores
scores_good = np.concatenate([
    np.random.normal(0.8, 0.12, 75),
    np.random.normal(0.2, 0.12, 75)
])
scores_good = np.clip(scores_good, 0.0, 1.0)

# Poor Classifier Scores
scores_poor = np.concatenate([
    np.random.normal(0.55, 0.2, 75),
    np.random.normal(0.45, 0.2, 75)
])
scores_poor = np.clip(scores_poor, 0.0, 1.0)

## 2. Calculating the ROC Curve from Scratch

Let's write a function to evaluate TPR and FPR at all possible threshold settings.
Steps:
1. Sort thresholds descending (from $1.0$ down to $0.0$).
2. For each threshold, evaluate y_pred = (scores $\ge$ threshold).
3. Compute TP, TN, FP, FN.
4. Calculate $TPR = TP / (TP + FN)$ and $FPR = FP / (TN + FP)$.

In [ ]:
def custom_roc_curve(y_true, scores):
    """
    Calculate ROC curve coordinates (FPR, TPR) from scratch.
    """
    thresholds = np.sort(scores)[::-1]
    thresholds = np.concatenate([[1.001], thresholds])
    
    tprs = []
    fprs = []
    
    for thresh in thresholds:
        y_pred = (scores >= thresh).astype(int)
        
        TP = np.sum((y_true == 1) & (y_pred == 1))
        TN = np.sum((y_true == 0) & (y_pred == 0))
        FP = np.sum((y_true == 0) & (y_pred == 1))
        FN = np.sum((y_true == 1) & (y_pred == 0))
        
        tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        fpr = FP / (TN + FP) if (TN + FP) > 0 else 0.0
        
        tprs.append(tpr)
        fprs.append(fpr)
        
    return np.array(fprs), np.array(tprs), thresholds

# Calculate custom ROC curves
fpr_good, tpr_good, thresh_good = custom_roc_curve(y_true, scores_good)
fpr_poor, tpr_poor, thresh_poor = custom_roc_curve(y_true, scores_poor)

# Verify against sklearn
fpr_sk, tpr_sk, _ = roc_curve(y_true, scores_good)
print("FPR curves close?", np.allclose(np.interp(fpr_sk, fpr_good, fpr_good), fpr_sk))

## 3. Visualizing the ROC Curves

Let's plot both ROC curves to see the visual difference.

In [ ]:
plt.figure(figsize=(10, 7))

plt.plot(fpr_good, tpr_good, color='forestgreen', linewidth=3, label='Good Classifier')
plt.plot(fpr_poor, tpr_poor, color='darkorange', linewidth=2.5, linestyle='-.', label='Poor Classifier')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Guess (AUC = 0.50)')

# Annotate specific threshold points on the Good Classifier Curve
indices_to_label = [len(thresh_good)//4, len(thresh_good)//2, 3*len(thresh_good)//4]
for idx in indices_to_label:
    t = thresh_good[idx]
    f = fpr_good[idx]
    p = tpr_good[idx]
    plt.scatter(f, p, color='red', s=60, zorder=5)
    plt.annotate(f"Thresh={t:.2f}", (f+0.02, p-0.03), fontsize=10)

plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR / Recall)')
plt.title('ROC Curves: Good vs. Poor Separation')
plt.xlim(-0.02, 1.02)
plt.ylim(-0.02, 1.02)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.show()

## 💡 ROC Curve vs. PR Curve (Conceptual Check)
Why does computer vision prefer **Precision-Recall (PR) Curves** over **ROC Curves** for object detection?
1.  **Sensitivity to Class Imbalance:** ROC curves are insensitive to class imbalance because the False Positive Rate ($FPR = FP / (TN + FP)$) includes True Negatives ($TN$) in its denominator. In object detection, the background area ($TN$) is virtually infinite. Therefore, even if a model makes thousands of false alarms ($FP$), the denominator is so large that $FPR \approx 0$. This forces the ROC curve to look nearly perfect even if the model performs poorly!
2.  **PR Curve Behavior:** The Precision formula ($TP / (TP + FP)$) ignores True Negatives entirely. If the model makes false alarms, Precision drops immediately, giving an accurate representation of detection performance under class imbalance.